# Generalization
**Report section:** Generalization & error analysis (20 pts)

In [1]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

QUICK = False  # True = 2,000-row debug run, False = full data (use for the report)

In [2]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import SEED, set_seed, RESULTS_DIR
from src.data import get_split, load_test, load_unlabeled, xy
from src.features import build_features
from src.models import make_model
from src.evaluate import (auroc, cross_val_auroc, predict_proba1, plot_roc, savefig,
                          log_result, save_predictions)
set_seed(SEED)

## Data (already cleaned)
`src/data.py` fixes encoding and applies NFC. Fixed split: 8,000 train / 2,000 val (stratified, seed 42), same for everyone.
**Current best:** `make_model("robust_svm_tuned_blank")`, val 0.991, Kaggle public 0.913.
Reference: `baseline` (word+char TF-IDF + LogReg) val 0.956, Kaggle 0.860.

In [3]:
train_df, val_df = get_split(quick=QUICK)   # 8,000 / 2,000 rows, 50% AI each
test_df = load_test()                        # 3,000 rows (never dropped)
unl_df = load_unlabeled()                    # 3,000 rows, no labels
X_tr, y_tr = xy(train_df)
X_va, y_va = xy(val_df)

[train] raw=10000 empty=0 dup_rows=0 conflicting_label_rows=0 dropped=0 final=10000 (clean=True)
split: train=8000 (AI=0.500)  val=2000 (AI=0.500)  quick=False clean=True


[test] raw=3000 empty=0 dup_rows=0 conflicting_label_rows=0 dropped=0 final=3000 (clean=True)


[unlabeled] raw=3000 empty=0 dup_rows=0 conflicting_label_rows=0 dropped=0 final=3000 (clean=True)


## Experiments to run (report Q4.1-Q4.2)
1. **Q4.1:** train vs val AUROC of `robust_svm_tuned_blank`; explain the gap.
2. **Q4.2:** val AUROC by **>= 3 length bins** (e.g. <150, 150-250, >250 words) and by line-break style
   (`has_blank_line_breaks`, single, none). Explain the grouping choice.
3. Robustness: val AUROC after removing line breaks / adding typos to AI texts (our proxy for Kaggle).
4. Adversarial validation (train vs test, analysis only): which features shift? (poems are 3.6% of train, 18% of test.)

**Logging:** `log_result("<experiment>", config, metrics)` writes to `results/`, and
`savefig(fig, "<name>")` writes to `figures/`.

In [4]:
import re
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src.evaluate import rank_average
from src.features import STYLO_FEATURES, StylometricFeatures, has_blank_line_breaks, normalize_style, word_tfidf

SAVE_OUTPUTS = False  # True also writes the tables below to results/
KAGGLE_PUBLIC = {"SVM + blank push": 0.91328, "transformer": 0.88599, "SVM + transformer (rank average)": 0.91475}
PUSH = 0.1            # same blank-line push as robust_svm_tuned_blank


def safe_auroc(y, p):
    # AUROC is undefined for a group containing only one class.
    return auroc(y, p) if len(np.unique(y)) == 2 else np.nan


def blank_push(p, texts):
    b = np.array([has_blank_line_breaks(t) for t in texts], dtype=float)
    return (p + PUSH * b) / (1 + PUSH)


def saved_preds(path, split, ids):
    d = pd.read_csv(path)
    return d[d["split"] == split].set_index("id").loc[ids, "prob"].to_numpy()


def save_table(df, name):
    if SAVE_OUTPUTS and not QUICK:
        df.to_csv(RESULTS_DIR / f"{name}.csv", index=False)


t0 = time.time()
svm_blank = make_model("robust_svm_tuned_blank").fit(X_tr, y_tr)
svm_plain = svm_blank.estimator_
print(f"SVM fitted in {time.time() - t0:.0f}s")

p_tr_svm, p_va_svm = predict_proba1(svm_plain, X_tr), predict_proba1(svm_plain, X_va)
p_tr_blank, p_va_blank = predict_proba1(svm_blank, X_tr), predict_proba1(svm_blank, X_va)

tf_file = RESULTS_DIR / "preds_transformer.csv"
p_va_tf = saved_preds(tf_file, "val", val_df["id"])
p_te_tf = saved_preds(tf_file, "test", test_df["id"])
p_te_svm = predict_proba1(svm_plain, test_df["text"])

SVM fitted in 50s


## Q4.1: Train versus validation AUROC
The SVM is refit on the 8,000 training rows and scored on the training rows and the shared validation set. The 95% interval is a bootstrap over validation rows. Kaggle public scores are copied from the leaderboard.

In [5]:
def boot_ci(y, p, n=1000, seed=SEED):
    r = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    s = [safe_auroc(y[i], p[i]) for i in (r.integers(0, len(y), len(y)) for _ in range(n))]
    return np.nanpercentile(s, [2.5, 97.5])


p_va_ens = rank_average([p_va_blank, blank_push(p_va_tf, X_va)])
rows = []
for name, p_tr, p_va in [("SVM", p_tr_svm, p_va_svm), ("SVM + blank push", p_tr_blank, p_va_blank),
                         ("transformer", None, p_va_tf), ("SVM + transformer (rank average)", None, p_va_ens)]:
    lo, hi = boot_ci(y_va, p_va)
    rows.append({"model": name, "train_auroc": auroc(y_tr, p_tr) if p_tr is not None else np.nan,
                 "val_auroc": auroc(y_va, p_va), "val_ci_low": lo, "val_ci_high": hi,
                 "kaggle_public": KAGGLE_PUBLIC.get(name, np.nan)})
q41 = pd.DataFrame(rows)
q41["train_minus_val"] = q41.train_auroc - q41.val_auroc
q41["val_minus_kaggle"] = q41.val_auroc - q41.kaggle_public
save_table(q41, "gen_q41_train_val")
q41.round(4)

,model,train_auroc,val_auroc,val_ci_low,val_ci_high,kaggle_public,train_minus_val,val_minus_kaggle
0,SVM,1.0,0.9786,0.9732,0.9834,NaN,0.0214,NaN
1,SVM + blank push,1.0,0.9905,0.9868,0.9939,0.9133,0.0095,0.0773
2,transformer,NaN,0.9849,0.9805,0.9890,0.8860,NaN,0.0989
3,SVM + transformer (rank average),NaN,0.9949,0.9924,0.9970,0.9148,NaN,0.0801


**What I found:** the SVM is basically perfect on train (1.000) and gets 0.979 on val, so about a 0.021 gap. With the blank-line push val goes up to 0.9905 and the gap is about 0.010. Some gap is expected, since a linear model on a huge sparse feature set can separate 8,000 training rows almost perfectly.

The bigger problem is val to Kaggle. The SVM with the push drops from 0.9905 to 0.9133 (0.077) and the transformer drops from 0.9849 to 0.8860 (0.099). So overfitting to the training rows isn't the main issue, the test set just looks different from what we trained on.

## Q4.2: Validation AUROC by group
**Length** uses three bins around the roughly 200-word average, since shorter passages give a model less evidence. **Line-break style** separates blank-line paragraphs, single line breaks and no breaks, because layout is the clearest known difference between our data sources. The second table shows how common each group is in the validation and test sets, and how closely the SVM and transformer agree on test texts in that group.

In [6]:
def length_bin(t):
    n = len(t.split())
    return "<150 words" if n < 150 else "150-250 words" if n <= 250 else ">250 words"


def break_style(t):
    return "blank lines" if has_blank_line_breaks(t) else "single breaks" if "\n" in t else "no breaks"


GROUPINGS = {"length": length_bin, "line breaks": break_style}
va_scores = {"SVM": p_va_svm, "SVM + blank push": p_va_blank, "transformer": p_va_tf}
rows, share_rows = [], []
for grouping, fn in GROUPINGS.items():
    g_va = np.array([fn(t) for t in X_va])
    g_te = np.array([fn(t) for t in test_df["text"]])
    for g in sorted(set(g_va) | set(g_te)):
        m = g_va == g
        row = {"grouping": grouping, "group": g, "n_val": int(m.sum()), "frac_ai": y_va[m].mean() if m.any() else np.nan}
        row.update({k: safe_auroc(y_va[m], v[m]) if m.any() else np.nan for k, v in va_scores.items()})
        rows.append(row)
        mt = g_te == g
        share_rows.append({"grouping": grouping, "group": g, "share_val": m.mean(), "share_test": mt.mean(),
                           "test_svm_vs_transformer_spearman": spearmanr(p_te_svm[mt], p_te_tf[mt]).statistic if mt.sum() > 2 else np.nan})
q42 = pd.DataFrame(rows)
q42_share = pd.DataFrame(share_rows)
save_table(q42, "gen_q42_groups")
save_table(q42_share, "gen_q42_group_shares")
display(q42.round(4))
q42_share.round(3)

,grouping,group,n_val,frac_ai,SVM,SVM + blank push,transformer
0,length,150-250 words,1065,0.4131,0.9805,0.9939,0.9845
1,length,<150 words,404,0.6287,0.9601,0.9754,0.9828
2,length,>250 words,531,0.5763,0.9883,0.9967,0.9883
3,line breaks,blank lines,562,1.0000,NaN,NaN,NaN
4,line breaks,no breaks,602,0.4419,0.9766,0.9766,0.9809
5,line breaks,single breaks,836,0.2057,0.9876,0.9876,0.9792


,grouping,group,share_val,share_test,test_svm_vs_transformer_spearman
0,length,150-250 words,0.532,0.551,0.741
1,length,<150 words,0.202,0.138,0.798
2,length,>250 words,0.266,0.311,0.850
3,line breaks,blank lines,0.281,0.213,0.632
4,line breaks,no breaks,0.301,0.365,0.765
5,line breaks,single breaks,0.418,0.422,0.607


**What I found:**
- Short texts are the hardest for the SVM: 0.960 under 150 words vs 0.988 over 250. The transformer is a lot flatter (0.983 vs 0.988).
- All 562 blank-line texts in val are AI, so AUROC is undefined inside that group. The push only helps by ranking those texts above everything else, which is why the no-break and single-break numbers don't change with it.
- Test actually has fewer blank-line texts (21% vs 28%) and fewer short texts (14% vs 20%), so the mix of these groups doesn't explain the Kaggle drop by itself.
- The SVM and transformer agree least on test texts with blank lines (0.63) and single line breaks (0.61), and most on long texts (0.85).

## Robustness to edited AI text
Validation AI texts are edited and human texts are left alone, as a stand-in for AI text that was cleaned up or lightly rewritten. The transformer is not included because it needs a GPU.

In [7]:
def remove_breaks(t):
    return re.sub(r"\s*\n\s*", " ", t)


def add_typos(t, r, rate=0.05):
    words = t.split(" ")
    for i, w in enumerate(words):
        if len(w) > 3 and w.isalpha() and r.random() < rate:
            j = int(r.integers(1, len(w) - 1))
            words[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]
    return " ".join(words)


def perturb_ai(fn):
    return [fn(t) if lab == 1 else t for t, lab in zip(X_va, y_va)]


r = np.random.default_rng(SEED)
variants = {"original": list(X_va), "AI: line breaks removed": perturb_ai(remove_breaks),
            "AI: typos added": perturb_ai(lambda t: add_typos(t, r)),
            "AI: both": perturb_ai(lambda t: add_typos(remove_breaks(t), r))}
robust = pd.DataFrame([{"variant": k, "SVM": auroc(y_va, predict_proba1(svm_plain, v)),
                        "SVM + blank push": auroc(y_va, predict_proba1(svm_blank, v))} for k, v in variants.items()])
save_table(robust, "gen_robustness")
robust.round(4)

,variant,SVM,SVM + blank push
0,original,0.9786,0.9905
1,AI: line breaks removed,0.9786,0.9786
2,AI: typos added,0.9760,0.9895
3,AI: both,0.9762,0.9762


**What I found:** removing line breaks from the AI texts wipes out the whole push (0.9905 to 0.9786). The SVM alone doesn't change at all, since its features already ignore whitespace. Typos barely do anything (0.9786 to 0.9760). So the model holds up against these edits, which means they probably aren't what's costing us on Kaggle.

## Adversarial validation: how different is the test set?
A classifier tries to tell labeled rows from test rows. An AUROC near 0.5 would mean the two look alike. Test rows are used only for this analysis, never for training a submission model.

In [8]:
X_adv = list(X_tr) + list(X_va) + test_df["text"].tolist()
y_adv = np.r_[np.zeros(len(X_tr) + len(X_va), dtype=int), np.ones(len(test_df), dtype=int)]
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

S = StylometricFeatures(robust=False).fit_transform(X_adv)
p_style = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), S, y_adv, cv=cv, method="predict_proba")[:, 1]
tr_m, te_m = S[y_adv == 0], S[y_adv == 1]
shift = pd.DataFrame({"feature": STYLO_FEATURES, "labeled_mean": tr_m.mean(0), "test_mean": te_m.mean(0),
                      "std_diff": (te_m.mean(0) - tr_m.mean(0)) / S.std(0).clip(1e-9)})
shift = shift.reindex(shift.std_diff.abs().sort_values(ascending=False).index).head(10)

vec = word_tfidf(ngram_range=(1, 1), min_df=5, preprocessor=lambda t: normalize_style(t).lower())
W = vec.fit_transform(X_adv)
clf = LogisticRegression(C=1.0, max_iter=2000, solver="liblinear")
p_word = cross_val_predict(clf, W, y_adv, cv=cv, method="predict_proba")[:, 1]
coef = clf.fit(W, y_adv).coef_[0]
vocab = vec.get_feature_names_out()
order = np.argsort(coef)

print("adversarial AUROC, style features:", round(auroc(y_adv, p_style), 4))
print("adversarial AUROC, word TF-IDF:  ", round(auroc(y_adv, p_word), 4))
print("words most typical of test:   ", ", ".join(vocab[order[-20:][::-1]]))
print("words most typical of labeled:", ", ".join(vocab[order[:20]]))
save_table(shift, "gen_adversarial_style")
shift.round(3)

adversarial AUROC, style features: 0.6991
adversarial AUROC, word TF-IDF:   0.7349
words most typical of test:    his, novel, ,, the, a, book, love, begins, young, becomes, poem, heart, finds, untill, ;, come, sky, friends, soul, friend
words most typical of labeled: film, was, (, ), overall, 1, ., /, movie, t, 2, hey, experience, trying, fucking, have, born, season, according, guy


,feature,labeled_mean,test_mean,std_diff
7,comma_rate,0.045,0.060,0.439
21,stopword_ratio,0.425,0.462,0.336
19,digit_ratio,0.013,0.007,-0.321
14,paren_rate,0.014,0.008,-0.244
3,n_sents,15.606,18.260,0.224
22,newline_count,8.977,12.450,0.218
2,avg_word_len,4.770,4.648,-0.182
8,semicolon_rate,0.001,0.002,0.172
5,std_sent_len,9.625,7.895,-0.167
10,em_dash_rate,0.000,0.000,0.158


**What I found:** a classifier can tell test from labeled rows with AUROC 0.70 on style features and 0.73 on words, so the shift is real but moderate. The words point to a genre change: test leans toward novel, book, love, poem, heart, while the labeled data leans toward film, movie, overall, season. Test also has more commas and stopwords and fewer digits and parentheses.

My best guess is that the training data is heavier on reviews and web text while test has more literature and poetry, which fits poems being 3.6% of train but 18% of test. I can't confirm that's what is hurting the score without test labels.